In [ ]:
import nico 
from nico import Annotations as sann
from nico import Interactions as sint
from nico import Covariations as scov

import numpy as np
import os
import matplotlib.pyplot as plt 
from matplotlib.collections import PatchCollection

In [ ]:
import igraph
import scanpy as sc
import shutil
import pandas as pd

In [ ]:
data_normal = sc.read("/cluster3/yflu/STS/TMA/TMA_normal_pegasus_250823.h5ad")
data_T = sc.read("/cluster3/yflu/STS/TMA/TMA_T_pegasus_250820.h5ad")
data_B = sc.read("/cluster3/yflu/STS/TMA/TMA_B_pegasus_250822.h5ad")
data_M = sc.read("/cluster3/yflu/STS/TMA/TMA_M_pegasus_250822.h5ad")

In [ ]:
data_normal.obs['Celltype_united']=data_normal.obs.celltype.copy()
data_normal.obs.Celltype_united.replace(['T cells'],'T/NK cells',inplace=True)
data_normal.obs.Celltype_united.replace(['Hepatocytes'],'Other cells',inplace=True)
data_normal.obs.Celltype_united.replace(['Liver sinusoidal endothelial cells'],'Other cells',inplace=True)
data_normal.obs.Celltype_united.replace(['Neural progenitors'],'Other cells',inplace=True)
data_normal.obs.Celltype_united.replace(['Adipocytes'],'Other cells',inplace=True)

In [ ]:
data_normal.obs.Celltype_united.value_counts()

In [ ]:
sample = "T1642"

In [ ]:
output_nico_dir=f'/cluster3/yflu/STS/TMA/separated/{sample}/'
output_annotation_dir=None #uses default location
#output_annotation_dir=output_nico_dir+'annotations/'
annotation_save_fname= f'{sample}_nico_1.h5ad'
inputRadius=0

In [ ]:
adata = sc.read(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_annotated.h5ad")

In [ ]:
adata.obs.celltype

In [ ]:
adata = sc.read(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_annotated.h5ad")

In [ ]:
adata.obs.index = adata.obs.cell_id

In [ ]:
data_normal_sub = data_normal[data_normal.obs["Sample"] == sample].copy()
data_T_sub = data_T[data_T.obs["Sample"] == sample].copy()
data_B_sub = data_B[data_B.obs["Sample"] == sample].copy()
data_M_sub = data_M[data_M.obs["Sample"] == sample].copy()

In [ ]:
data_normal_sub.obs.index = data_normal_sub.obs.index.str[5:] + "-1"
data_T_sub.obs.index = data_T_sub.obs.index.str[5:] + "-1"
data_B_sub.obs.index = data_B_sub.obs.index.str[5:] + "-1"
data_M_sub.obs.index = data_M_sub.obs.index.str[5:] + "-1"

In [ ]:
# 先初始化为 adata.obs["celltype"]
adata.obs["celltype_sp"] = adata.obs["celltype"].astype(str)

# 把三个对象的 celltype_sp 合并成一个 Series
mapping = pd.concat([
    data_normal_sub.obs["celltype"],
    data_T_sub.obs["celltype_sp"],
    data_B_sub.obs["celltype_sp"],
    data_M_sub.obs["celltype_sp"]
])

# 只保留 adata 中有的细胞
mapping = mapping[mapping.index.isin(adata.obs_names)]

# 用映射一次性更新
adata.obs.loc[mapping.index, "celltype_sp"] = mapping

In [ ]:
# 先初始化为 adata.obs["celltype"]
adata.obs["Celltype_united"] = adata.obs["celltype"].astype(str)

# 把三个对象的 celltype_sp 合并成一个 Series
mapping = pd.concat([
    data_normal_sub.obs["Celltype_united"]
])

# 只保留 adata 中有的细胞
mapping = mapping[mapping.index.isin(adata.obs_names)]

# 用映射一次性更新
adata.obs.loc[mapping.index, "Celltype_united"] = mapping

In [ ]:
sc.pl.spatial(adata,color = ["celltype_sp","Celltype_united"],spot_size=10, show=False)
sc.pl.spatial(
    adata,
    color="celltype_sp",
    groups=["T endo-niche", "T fibro-niche","T mye-niche","Endothelial"],  # 只显示指定的类别
    spot_size=10,
    show=False
)

In [ ]:
adata.obs.celltype_sp.value_counts()

In [ ]:
adata.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_nico_1.h5ad")

In [ ]:
ct_counts = adata.obs["celltype_sp"].value_counts()

# 找出小于 50 的 celltype
do_not_use_following_CT_in_niche = ct_counts[ct_counts < 50].index.tolist()

print(do_not_use_following_CT_in_niche)

In [ ]:
niche_pred_output=sint.spatial_neighborhood_analysis(
Radius=inputRadius,
output_nico_dir=output_nico_dir,
anndata_object_name=annotation_save_fname,
spatial_cluster_tag='celltype_sp',
removed_CTs_before_finding_CT_CT_interactions=do_not_use_following_CT_in_niche)

In [ ]:
celltype_niche_interaction_cutoff=0.1

In [ ]:
sint.plot_niche_interactions_without_edge_weight(niche_pred_output,
niche_cutoff=celltype_niche_interaction_cutoff,
saveas="pdf",
transparent_mode=False,
showit=True,
figsize=(10,7),
input_colormap='jet',   #Colormap for node colors, from matplotlib colormaps.
with_labels=True,       #Display cell type labels on the nodes, if True.
node_size=300,          #Size of the nodes.
linewidths=0.5,         #Width of the node border lines.
node_font_size=6,       #Font size for node labels.
alpha=0.5,              #Opacity level for nodes and edges. 1 is fully opaque, and 0 is fully transparent.
font_weight='bold'      #Font weight for node labels; 'bold' for emphasis, 'normal' otherwise.
)

In [ ]:
sint.plot_niche_interactions_with_edge_weight(niche_pred_output,
niche_cutoff=celltype_niche_interaction_cutoff,
saveas="pdf",
showit=True,
figsize=(10,7),
input_colormap='jet',
with_labels=True,
node_size=500,
linewidths=1,
node_font_size=8,
alpha=0.5,
font_weight='normal',
edge_label_pos=0.35,   #Relative position of the weight label along the edge.
edge_font_size=8       #Font size for edge labels.
)

In [ ]:
niche_pred_output=sint.spatial_neighborhood_analysis(
Radius=inputRadius,
output_nico_dir=output_nico_dir,
anndata_object_name=annotation_save_fname,
spatial_cluster_tag='Celltype_united')

In [ ]:
sint.plot_niche_interactions_without_edge_weight(niche_pred_output,
niche_cutoff=celltype_niche_interaction_cutoff,
saveas="pdf",
transparent_mode=False,
showit=True,
figsize=(10,7),
input_colormap='jet',   #Colormap for node colors, from matplotlib colormaps.
with_labels=True,       #Display cell type labels on the nodes, if True.
node_size=300,          #Size of the nodes.
linewidths=0.5,         #Width of the node border lines.
node_font_size=6,       #Font size for node labels.
alpha=0.5,              #Opacity level for nodes and edges. 1 is fully opaque, and 0 is fully transparent.
font_weight='bold'      #Font weight for node labels; 'bold' for emphasis, 'normal' otherwise.
)

In [ ]:
sint.plot_niche_interactions_with_edge_weight(niche_pred_output,
niche_cutoff=celltype_niche_interaction_cutoff,
saveas="pdf",
showit=True,
figsize=(10,7),
input_colormap='jet',
with_labels=True,
node_size=500,
linewidths=1,
node_font_size=8,
alpha=0.5,
font_weight='normal',
edge_label_pos=0.35,   #Relative position of the weight label along the edge.
edge_font_size=8       #Font size for edge labels.
)

In [ ]:
adata.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/sct_spatial.h5ad")

In [ ]:
adata

In [ ]:
ref_datapath='/cluster3/yflu/STS/TMA/celltypist/'
query_datapath=f"/cluster3/yflu/STS/TMA/separated/{sample}/"

In [ ]:
cov_out=scov.gene_covariation_analysis(iNMFmode=True,
        Radius=inputRadius,
        no_of_factors=1,
        refpath=ref_datapath,
        quepath=query_datapath,
        ref_cluster_tag='Celltype_united',
        spatial_integration_modality='double',
        output_niche_prediction_dir=output_nico_dir,
        LRdbFilename='/cluster3/yflu/STS/TMA/NiCoLRdb.txt')

In [ ]:
scov.plot_cosine_and_spearman_correlation_to_factors(cov_out,
choose_celltypes=['T/NK cells'],
NOG_Fa=30,saveas="pdf",
figsize=(15,10))

In [ ]:
dataFrame=scov.extract_and_plot_top_genes_from_chosen_factor_in_celltype(
cov_out,
choose_celltype="B cells",
choose_factor_id=1,
top_NOG=20,
correlation_with_spearman=True,
positively_correlated=False,
saveas="pdf")

In [ ]:
choose_celltypes=["Endothelial"]
scov.plot_significant_regression_covariations_as_circleplot(cov_out,
choose_celltypes=choose_celltypes,
mention_pvalue=True,
saveas="pdf",
figsize=(6,1.25))

In [ ]:
scov.plot_significant_regression_covariations_as_heatmap(cov_out,
choose_celltypes=['Endothelial'],
saveas="pdf", figsize=(6,1.25))

In [ ]:
scov.save_LR_interactions_in_excelsheet_and_regression_summary_in_textfile_for_interacting_cell_types(cov_out,
pvalueCutoff=0.05,correlation_with_spearman=True,
LR_plot_NMF_Fa_thres=0.1,LR_plot_Exp_thres=0.1,number_of_top_genes_to_print=5)

In [ ]:
scov.find_LR_interactions_in_interacting_cell_types(cov_out,
choose_interacting_celltype_pair=[],
choose_factors_id=[1,1],
pvalueCutoff=0.05,
LR_plot_NMF_Fa_thres=0.1,
LR_plot_Exp_thres=0.1,
saveas="pdf",figsize=(12, 10))